# Transformer NLP: Classification + Text Generation

**Part A:** IMDb + DistilBERT  
**Part B:** ELI5 + DistilGPT2

Compact, Colab-ready notebook covering tokenization, fine-tuning, evaluation, and generation.

In [ ]:
!pip -q install -U transformers datasets evaluate accelerate

import math, numpy as np, torch
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM,
    DataCollatorWithPadding, DataCollatorForLanguageModeling,
    TrainingArguments, Trainer, pipeline
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 113.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 34.2 MB/s eta 0:00:00


## Part A — IMDb Sentiment Classification

IMDb contains positive (`1`) and negative (`0`) movie reviews. We tokenize the reviews, fine-tune DistilBERT for two classes, evaluate accuracy, and classify new reviews.

In [ ]:
# 1–2. Load and inspect IMDb
imdb = load_dataset("stanfordnlp/imdb")
print(imdb)
print(imdb["train"][0]["text"][:300], "->", imdb["train"][0]["label"])

# Set True for a faster demonstration
USE_SMALL_DATASET = False

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h -> 0


In [ ]:
# 3–4. Tokenization, truncation and padding
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

def tokenize(x):
    return tokenizer(x["text"], truncation=True)

tok = imdb.map(tokenize, batched=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
# 5–6. Load DistilBERT and fine-tune
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased",
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1}
)

train_ds, test_ds = tok["train"], tok["test"]
if USE_SMALL_DATASET:
    train_ds = train_ds.shuffle(seed=42).select(range(2000))
    test_ds = test_ds.shuffle(seed=42).select(range(1000))

accuracy = evaluate.load("accuracy")
def metrics(p):
    pred, labels = p
    return accuracy.compute(
        predictions=np.argmax(pred, axis=1), references=labels
    )

args = TrainingArguments(
    output_dir="./imdb-distilbert",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=test_ds,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=metrics
)
trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.223758,0.203955,0.922360
2,0.147575,0.234198,0.931160


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3126, training_loss=0.20671456300975874, metrics={'train_runtime': 913.6778, 'train_samples_per_second': 54.724, 'train_steps_per_second': 3.421, 'total_flos': 6556904415524352.0, 'train_loss': 0.20671456300975874, 'epoch': 2.0})

In [ ]:
# 7–8. Evaluate and predict
result = trainer.evaluate()
print(f"Test accuracy: {result['eval_accuracy']:.4f}")

clf = pipeline("sentiment-analysis", model=trainer.model, tokenizer=tokenizer)

reviews = [
    "The acting was brilliant and the story was engaging.",
    "The movie was boring, poorly written, and a waste of time."
]
for r in reviews:
    print(r, "->", clf(r))

Training Loss,Validation Loss,Epoch,Accuracy
0.147575,0.203955,2,0.922360


Test accuracy: 0.9224
The acting was brilliant and the story was engaging. -> [{'label': 'POSITIVE', 'score': 0.9889918565750122}]
The movie was boring, poorly written, and a waste of time. -> [{'label': 'NEGATIVE', 'score': 0.9944666028022766}]


## Part B — Causal Language Modeling

ELI5 answer text is tokenized and grouped into fixed-length blocks. DistilGPT2 learns to predict the next token (`mlm=False`). We then calculate perplexity and generate text with sampling parameters.

In [ ]:
# 1. Load a manageable ELI5 subset
eli5 = load_dataset("dany0407/eli5_category", split="train[:5000]")
eli5 = eli5.train_test_split(test_size=0.2, seed=42).flatten()
print(eli5)

README.md:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 98.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation1-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.92MB            

data/validation1-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation2-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.78MB            

... (4 outputs left out)

Generating validation1 split:   0%|          | 0/5446 [00:00<?, ? examples/s]

Generating validation2 split:   0%|          | 0/2375 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5411 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['q_id', 'title', 'selftext', 'category', 'subreddit', 'answers.a_id', 'answers.text', 'answers.score', 'answers.text_urls', 'title_urls', 'selftext_urls'],
        num_rows: 4000
    })
    test: Dataset({
        features: ['q_id', 'title', 'selftext', 'category', 'subreddit', 'answers.a_id', 'answers.text', 'answers.score', 'answers.text_urls', 'title_urls', 'selftext_urls'],
        num_rows: 1000
    })
})


In [ ]:
# 2–3. Tokenize answers and group into 128-token blocks
tokenizer_gpt = AutoTokenizer.from_pretrained("distilbert/distilgpt2")
tokenizer_gpt.pad_token = tokenizer_gpt.eos_token

def tokenize_answers(x):
    texts = [" ".join(a) for a in x["answers.text"]]
    return tokenizer_gpt(texts)

tok_eli5 = eli5.map(
    tokenize_answers,
    batched=True,
    num_proc=1,
    remove_columns=eli5["train"].column_names
)

block_size = 128

def group_texts(x):
    joined = {k: sum(x[k], []) for k in x}
    n = (len(joined["input_ids"]) // block_size) * block_size
    out = {
        k: [v[i:i+block_size] for i in range(0, n, block_size)]
        for k, v in joined.items()
    }
    out["labels"] = out["input_ids"].copy()
    return out

lm_data = tok_eli5.map(group_texts, batched=True, num_proc=1)
print(lm_data)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map (num_proc=1):   0%|          | 0/4000 [00:00<?, ? examples/s]

... (2 outputs left out)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1576 > 1024). Running this sequence through the model will result in indexing errors


Map (num_proc=1):   0%|          | 0/4000 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10463
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2619
    })
})


In [ ]:
# 4–6. Collator, DistilGPT2 and fine-tuning
collator_gpt = DataCollatorForLanguageModeling(
    tokenizer=tokenizer_gpt, mlm=False
)

model_gpt = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2")
model_gpt.config.pad_token_id = tokenizer_gpt.pad_token_id

args_gpt = TrainingArguments(
    output_dir="./eli5-distilgpt2",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer_gpt = Trainer(
    model=model_gpt,
    args=args_gpt,
    train_dataset=lm_data["train"],
    eval_dataset=lm_data["test"],
    data_collator=collator_gpt,
    processing_class=tokenizer_gpt
)
trainer_gpt.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.891791,3.819672


... (3 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2616, training_loss=3.9251059085950937, metrics={'train_runtime': 356.9245, 'train_samples_per_second': 29.314, 'train_steps_per_second': 7.329, 'total_flos': 341743487680512.0, 'train_loss': 3.9251059085950937, 'epoch': 1.0})

In [ ]:
# 7. Evaluate: perplexity = exp(evaluation loss)
ev = trainer_gpt.evaluate()
ppl = math.exp(ev["eval_loss"])
print(f"Evaluation loss: {ev['eval_loss']:.4f}")
print(f"Perplexity: {ppl:.2f}")

Training Loss,Validation Loss,Epoch
3.891791,3.819672,1


Evaluation loss: 3.8197
Perplexity: 45.59


In [ ]:
# 8–9. Generate text and compare top_k / top_p
generator = pipeline(
    "text-generation",
    model=trainer_gpt.model,
    tokenizer=tokenizer_gpt
)

prompt = "Why do humans dream?"

for top_k, top_p in [(20, 0.8), (50, 0.95), (100, 0.95)]:
    out = generator(
        prompt,
        max_new_tokens=60,
        do_sample=True,
        top_k=top_k,
        top_p=top_p,
        temperature=0.8,
        pad_token_id=tokenizer_gpt.eos_token_id
    )
    print(f"\ntop_k={top_k}, top_p={top_p}")
    print(out[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'temperature', 'top_k', 'max_new_tokens', 'pad_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyw


top_k=20, top_p=0.8
Why do humans dream? We do. If we do, we have to think about what we can and can't dream about. If we do, we have to think about what we can and cannot dream about. If we do, we have to think about what we can and cannot dream about. If we do, we


[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



top_k=50, top_p=0.95
Why do humans dream? It's a hard question because we don't really know, how does it work? But it is possible. Humans dream. We dream. In that way, we don't want to believe in anything that makes us dream. We don't want to believe in anything that's not real, or even

top_k=100, top_p=0.95
Why do humans dream? No. It's a simple question. The answer is not, as a physicist told me, as a physicist I never dreamed about thinking about it. The brain states that if we dream, it will be the same thing that we dream about imagining, so it will be the same thing. That
